# 02 Generate Synthetic Data And Classifier Experiments

This is the authoritative Task 1 runner. It retrains a GAN per data budget, generates the matching synthetic pool, and writes standardized outputs to `runs/task1/`. The shared pipeline now targets a local CPU-only setup for your MacBook workflow, so use the local override dict below if you want shorter runs while iterating.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from pipeline import DEFAULT_TASK1_SCENARIOS, DEFAULT_TASK1_SIZES, run_task1_pipeline

ROOT


In [ ]:
# Edit these values before running.
cfg = Config().with_runtime_profile_defaults()
DATA_ROOT = ROOT / "data_final"
TASK1_OUT_ROOT = ROOT / "runs" / "task1"
TASK1_SIZES = DEFAULT_TASK1_SIZES
TASK1_SCENARIOS = DEFAULT_TASK1_SCENARIOS
SYNTH_BATCH_SIZE = 64
STRICT_FID = True
NUM_WORKERS_OVERRIDE = None
LOCAL_TASK1_OVERRIDES = {}  # e.g. {"gan_epochs": 25, "clf_epochs": 10, "gan_batch": 16, "clf_batch": 16}

overrides = dict(LOCAL_TASK1_OVERRIDES)
if NUM_WORKERS_OVERRIDE is not None:
    overrides["num_workers"] = NUM_WORKERS_OVERRIDE
if overrides:
    cfg = cfg.with_overrides(**overrides)

{
    "cfg": cfg,
    "device": "cpu",
    "loader_options": cfg.loader_options(),
    "data_root": DATA_ROOT,
    "task1_out_root": TASK1_OUT_ROOT,
    "task1_sizes": TASK1_SIZES,
    "task1_scenarios": TASK1_SCENARIOS,
    "synth_batch_size": SYNTH_BATCH_SIZE,
    "strict_fid": STRICT_FID,
}


## Outputs

A full run writes:

- `runs/task1/gan/n*/...`
- `runs/task1/synth/n*/...`
- `runs/task1/clf/all_results.json`
- `runs/task1/pipeline_summary.json`


In [ ]:
task1_run = run_task1_pipeline(
    cfg,
    sizes=TASK1_SIZES,
    scenarios=TASK1_SCENARIOS,
    data_root=DATA_ROOT,
    out_root=TASK1_OUT_ROOT,
    synth_batch_size=SYNTH_BATCH_SIZE,
    strict_fid=STRICT_FID,
)

{
    "n_results": len(task1_run["all_results"]),
    "n_pipeline_rows": len(task1_run["pipeline_summary"]),
    "out_root": task1_run["out_root"],
}


In [ ]:
pipeline_summary = task1_run["pipeline_summary"]
pipeline_summary[-1] if pipeline_summary else {}